1. Carga de datos:

In [ ]:
import importlib.util
import sys
import os

# Definir ruta física
path_to_file = os.path.abspath('../src/retail-sales-analysis_P3.py')

# Cargar módulo manualmente
spec = importlib.util.spec_from_file_location("retail_module", path_to_file)
retail_module = importlib.util.module_from_spec(spec)
sys.modules["retail_module"] = retail_module
spec.loader.exec_module(retail_module)

# extraer funciones
cargar_datos = retail_module.cargar_datos
calcular_puntos = retail_module.calcular_puntos # <-- Agrega esta línea

print("¡Conexión establecida con el archivo .py!")

# Definir ruta del dataset y cargar Dataframe
ruta_archivo = '../data/retail_sales_dataset.csv'
df = cargar_datos(ruta_archivo) # Definir Dataframe

¡Conexión establecida con el archivo .py!


2. Exploración inicial:

In [2]:
# Inspeccionar dimensiones y tipos de datos
print(f'Dataset cargado exitosamente!')
print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Memoria aproximada: {df.memory_usage(deep=True).sum() / 1024:.0f} KB')
print(f'\nColumnas: {list(df.columns)}')

Dataset cargado exitosamente!
Dimensiones: 1000 filas x 16 columnas
Memoria aproximada: 405 KB

Columnas: ['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age', 'Product Category', 'Quantity', 'Price per Unit', 'Total Amount', 'Ingreso Total', 'Ventas Normalizadas', 'Categoria Venta', 'Mes', 'Rango_Etario', 'Desviacion_Media', 'Puntos_Lealtad']


4. Chequear extremos del dataset

In [3]:
# Visualizar primeros y últimos lugares en el dataset
print("\nPrimeros 5 registros:")
print(df.head(5))
print("\nÚltimos 5 registros:")
print(df.tail(5))


Primeros 5 registros:
   Transaction ID        Date Customer ID  Gender  Age Product Category  \
0               1  2023-11-24     CUST001    Male   34           Beauty   
1               2  2023-02-27     CUST002  Female   26         Clothing   
2               3  2023-01-13     CUST003    Male   50      Electronics   
3               4  2023-05-21     CUST004    Male   37         Clothing   
4               5  2023-05-06     CUST005    Male   30           Beauty   

   Quantity  Price per Unit  Total Amount  Ingreso Total  Ventas Normalizadas  \
0         3              50           150            150                 0.06   
1         2             500          1000           1000                 0.49   
2         1              30            30             30                 0.00   
3         1             500           500            500                 0.24   
4         2              50           100            100                 0.04   

  Categoria Venta  Mes Rango_Etario  De

In [4]:
print("=== 5 últimas filas ===")
print()
display(df.info(5))
print("No se encontraron valores nulos.")
print("================================")

=== 5 últimas filas ===

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Transaction ID       1000 non-null   int64  
 1   Date                 1000 non-null   str    
 2   Customer ID          1000 non-null   str    
 3   Gender               1000 non-null   str    
 4   Age                  1000 non-null   int64  
 5   Product Category     1000 non-null   str    
 6   Quantity             1000 non-null   int64  
 7   Price per Unit       1000 non-null   int64  
 8   Total Amount         1000 non-null   int64  
 9   Ingreso Total        1000 non-null   int64  
 10  Ventas Normalizadas  1000 non-null   float64
 11  Categoria Venta      1000 non-null   str    
 12  Mes                  1000 non-null   int64  
 13  Rango_Etario         1000 non-null   str    
 14  Desviacion_Media     1000 non-null   float64
 15  Puntos_Lealtad       1000

None

No se encontraron valores nulos.


In [5]:
print("=== Estadísticas descriptivas ===")
print()
display(df.describe().round(2))
print("=================================")

=== Estadísticas descriptivas ===



,Transaction ID,Age,Quantity,Price per Unit,Total Amount,Ingreso Total,Ventas Normalizadas,Mes,Desviacion_Media,Puntos_Lealtad
count,1000.00,1000.00,1000.00,1000.00,1000.0,1000.0,1000.00,1000.00,1000.00,1000.00
mean,500.50,41.39,2.51,179.89,456.0,456.0,0.22,6.55,0.00,49.78
std,288.82,13.68,1.13,189.68,560.0,560.0,0.28,3.45,559.91,85.39
min,1.00,18.00,1.00,25.00,25.0,25.0,0.00,1.00,-442.48,0.00
25%,250.75,29.00,1.00,30.00,60.0,60.0,0.02,4.00,-392.48,0.00
50%,500.50,42.00,3.00,50.00,135.0,135.0,0.06,6.00,-320.36,0.00
75%,750.25,53.00,4.00,300.00,900.0,900.0,0.44,10.00,432.52,90.00
max,1000.00,64.00,4.00,500.00,2000.0,2000.0,1.00,12.00,1556.75,300.00


### Desarrollo de instrucciones

1. Transformación de Datos
    * Crea nuevas columnas: Basándonos en los datos existentes, crea nuevas columnas que sean útiles para el análisis. Por ejemplo, calcula el ingreso total por venta y normaliza las ventas.
    * Clasifica los datos: Crea una columna que clasifique las ventas en categorías significativas (e.g., ‘Alta’, ‘Media’, ‘Baja’).


* Nuevas Columnas: 'Ingreso Total' y 'Ventas Normalizadas'

In [6]:
# Se define columna de Ingreso Total
df['Ingreso Total'] = df['Price per Unit'] * df['Quantity']

# Normalizar las ventas (Ingreso Total)
# Valores al rango [0, 1]
min_ingreso = df['Ingreso Total'].min()
max_ingreso = df['Ingreso Total'].max()

df['Ventas Normalizadas'] = ((df['Ingreso Total'] - min_ingreso) / (max_ingreso - min_ingreso)).round(2)

# Verificar nuevas columnas
print("Nuevas columnas creadas en el DataFrame 'df':")
display(df[['Price per Unit', 'Quantity', 'Ingreso Total', 'Ventas Normalizadas']].head())

Nuevas columnas creadas en el DataFrame 'df':


,Price per Unit,Quantity,Ingreso Total,Ventas Normalizadas
0,50,3,150,0.06
1,500,2,1000,0.49
2,30,1,30,0.00
3,500,1,500,0.24
4,50,2,100,0.04


* Calsificar Ventas

** Ver montos de ventas y su frecuencia


In [7]:
print("=== Montos de Ventas y su frecuencia ===")
print()
Contar_ventas = df['Ingreso Total'].value_counts().sort_index(ascending=False)
print(Contar_ventas)
print("Se definen las categorías de ventas:")
print("* Alta > 1000")
print("* 1000 >= Media > 100")
print("* 100 >= Baja")
print("========================================")


=== Montos de Ventas y su frecuencia ===

Ingreso Total
2000     49
1500     50
1200     54
1000     49
900      62
600      35
500      51
300      46
200      62
150      42
120      43
100     108
90       44
75       43
60       45
50      115
30       51
25       51
Name: count, dtype: int64
Se definen las categorías de ventas:
* Alta > 1000
* 1000 >= Media > 100
* 100 >= Baja


** Ejecutar función para categorizar las ventas

In [8]:
clasificar_ventas = retail_module.clasificar_ventas

# Ejecutar la clasificación
df = clasificar_ventas(df)

# Ver el resultado
print(df[['Ingreso Total', 'Categoria Venta']].head())

   Ingreso Total Categoria Venta
0            150           Media
1           1000           Media
2             30            Baja
3            500           Media
4            100            Baja



2. Agrupación y Agregación
    * Agrupación por múltiples columnas: Realiza agrupaciones por categorías como producto y tienda, producto y mes, etc.
    * Aplicar funciones de agregación: Utiliza funciones como sum, mean, count, min, max, std, y var para obtener estadísticas descriptivas de cada grupo.


* Agrupación por Categoría de Producto y Género

In [9]:
# Agrupación por Producto y Género
agrupado_genero = df.groupby(['Product Category', 'Gender'])['Total Amount'].agg(['sum', 'mean', 'count', 'min', 'max', 'std', 'var'])

print("Estadísticas por Producto y Género:")
display(agrupado_genero.round(2))

cat_lider = agrupado_genero['sum'].idxmax()[0]
genero_lider = agrupado_genero['sum'].idxmax()[1]
mayor_dispersion = agrupado_genero['std'].idxmax()

print(
    f"=== Hallazgos Estratégicos: Ventas por Categoría y Género ===\n"
    f"1. Dominancia de Mercado: La categoría {cat_lider} genera el mayor volumen de ingresos\n"
    f"   totales. Estrategia: Reforzar el stock y considerar esta categoría como el 'ancla'\n"
    f"   para promociones cruzadas con productos de menor rotación.\n\n"
    f"2. Perfil del Consumidor: El segmento {genero_lider} en {cat_lider} muestra el conteo de\n"
    f"   compras más alto. Táctica: Diseñar campañas de fidelización específicas para este\n"
    f"   perfil, ya que representan la base de ingresos más estable del negocio.\n\n"
    f"3. Análisis de Riesgo y Variabilidad: El grupo {mayor_dispersion[1]} en {mayor_dispersion[0]}\n"
    f"   presenta la mayor desviación estándar. Significado: El gasto en este nicho es errático\n"
    f"   y depende de compras impulsivas o de alto valor ocasional. Táctica: Implementar\n"
    f"   precios dinámicos o bundles para estabilizar el ticket promedio en este sector."
)

Estadísticas por Producto y Género:


sum    mean  count  min   max     std        var
Product Category Gender                                                    
Beauty           Female  74830  450.78    166   25  2000  538.74  290235.44
                 Male    68685  487.13    141   25  2000  592.90  351530.08
Clothing         Female  81275  467.10    174   25  2000  577.02  332948.03
                 Male    74305  419.80    177   25  2000  524.12  274697.83
Electronics      Female  76735  451.38    170   25  2000  548.64  301010.95
                 Male    80170  466.10    172   25  2000  587.13  344721.29

=== Hallazgos Estratégicos: Ventas por Categoría y Género ===
1. Dominancia de Mercado: La categoría Clothing genera el mayor volumen de ingresos
   totales. Estrategia: Reforzar el stock y considerar esta categoría como el 'ancla'
   para promociones cruzadas con productos de menor rotación.

2. Perfil del Consumidor: El segmento Female en Clothing muestra el conteo de
   compras más alto. Táctica: Diseñar campañas de fidelización específicas para este
   perfil, ya que representan la base de ingresos más estable del negocio.

3. Análisis de Riesgo y Variabilidad: El grupo Male en Beauty
   presenta la mayor desviación estándar. Significado: El gasto en este nicho es errático
   y depende de compras impulsivas o de alto valor ocasional. Táctica: Implementar
   precios dinámicos o bundles para estabilizar el ticket promedio en este sector.


* Agrupación por Categoría de Producto y Mes

In [10]:
import pandas as pd

# Convertir a formato fecha y extraer mes
df['Date'] = pd.to_datetime(df['Date'])
df['Mes'] = df['Date'].dt.month

# Agrupación por Producto y Mes
agrupado_mes = df.groupby(['Product Category', 'Mes'])['Total Amount'].agg(['sum', 'mean', 'count', 'min', 'max', 'std', 'var'])

print("Estadísticas por Producto y Mes:")
display(agrupado_mes.round(2))

# Variables dinámicas para la interpretación
cat_pico = agrupado_mes['sum'].idxmax()
mes_valle = agrupado_mes['sum'].idxmin()

print(
    f"=== Hallazgos Estratégicos: Ventas por Categoría y Mes ===\n"
    f"1. Estacionalidad Crítica: La categoría {cat_pico[0]} alcanzó su pico máximo de ingresos\n"
    f"   en el mes {cat_pico[1]}. Estrategia: Programar el abastecimiento de inventario\n"
    f"   con tres meses de antelación para este periodo y evitar quiebres de stock.\n\n"
    f"2. Optimización de Flujo: El punto más bajo de ventas se registra en {mes_valle[0]} (mes {mes_valle[1]}).\n"
    f"   Táctica: Implementar promociones de liquidación o eventos de 'ventas flash' durante\n"
    f"   estos meses valle para mantener la liquidez y reducir costos de almacenamiento.\n\n"
    f"3. Estabilidad del Ticket: Al comparar la media (mean) mes a mes, se observa si el valor\n"
    f"   percibido por el cliente se mantiene constante. Resultados: Si el promedio cae mientras\n"
    f"   el conteo (count) sube, el consumidor está respondiendo a ofertas de bajo margen.\n"
    f"   Táctica: Ajustar el mix de productos para proteger la rentabilidad unitaria."
)

Estadísticas por Producto y Mes:


sum    mean  count  min   max     std        var
Product Category Mes                                                    
Beauty           1    13930  535.77     26   25  2000  658.82  434041.38
                 2    14035  539.81     26   25  2000  697.75  486860.96
                 3    10545  502.14     21   25  1500  482.91  233206.43
                 4    11905  410.52     29   25  2000  519.65  270036.33
                 5    12450  444.64     28   30  2000  571.56  326675.79
                 6    10995  439.80     25   25  2000  512.25  262396.83
                 7    16090  595.93     27   30  2000  626.06  391955.84
                 8     9790  407.92     24   25  1200  451.73  204060.69
                 9     6320  316.00     20   25  2000  505.96  255998.95
                 10   15355  495.32     31   25  2000  552.58  305339.89
                 11    9700  388.00     25   25  2000  583.13  340039.58
                 12   12400  496.00     25   25  2000  583.34  340281.25
Clothing         1    13125  504.81     26   25  2000  540.87  292542.96
                 2    14560  441.21     33   25  2000  551.72  304396.92
                 3    15065  396.45     38   25  2000  497.56  247568.79
                 4    13940  387.22     36   25  2000  576.18  331987.78
                 5    17455  471.76     37   25  2000  567.65  322222.52
                 6    10170  363.21     28   25  1500  437.64  191528.17
                 7     8250  434.21     19   25  2000  602.11  362539.62
                 8    12455  389.22     32   25  2000  564.53  318692.11
                 9     9975  498.75     20   50  2000  630.42  397431.25
                 10   13315  443.83     30   25  2000  617.89  381790.83
                 11   15200  584.62     26   25  2000  565.79  320121.85
                 12   12070  464.23     26   30  2000  540.91  292585.38
Electronics      1     9925  381.73     26   25  2000  568.82  323559.88
                 2    15465  594.81     26   30  2000  623.95  389312.96
                 3     3380  241.43     14   30  1000  277.68   77105.49
                 4     8025  382.14     21   25  1500  512.22  262366.43
                 5    23245  581.12     40   25  2000  653.75  427387.80
                 6    15550  647.92     24   30  2000  675.03  455660.69
                 7    11125  427.88     26   30  2000  507.79  257852.35
                 8    14715  387.24     38   25  2000  509.45  259537.43
                 9     7325  293.00     25   30  2000  436.81  190804.17
                 10   17910  511.71     35   25  2000  662.39  438757.27
                 11   10020  371.11     27   25  1500  505.73  255758.33
                 12   20220  505.50     40   25  2000  560.32  313953.59

=== Hallazgos Estratégicos: Ventas por Categoría y Mes ===
1. Estacionalidad Crítica: La categoría Electronics alcanzó su pico máximo de ingresos
   en el mes 5. Estrategia: Programar el abastecimiento de inventario
   con tres meses de antelación para este periodo y evitar quiebres de stock.

2. Optimización de Flujo: El punto más bajo de ventas se registra en Electronics (mes 3).
   Táctica: Implementar promociones de liquidación o eventos de 'ventas flash' durante
   estos meses valle para mantener la liquidez y reducir costos de almacenamiento.

3. Estabilidad del Ticket: Al comparar la media (mean) mes a mes, se observa si el valor
   percibido por el cliente se mantiene constante. Resultados: Si el promedio cae mientras
   el conteo (count) sube, el consumidor está respondiendo a ofertas de bajo margen.
   Táctica: Ajustar el mix de productos para proteger la rentabilidad unitaria.


* Agrupación por Categoría de producto y rango Etario

In [11]:
# Crear rangos de edad (bins)
df['Rango_Etario'] = pd.cut(df['Age'], bins=[0, 25, 45, 100], labels=['Joven', 'Adulto', 'Senior'])

# Agrupación por Producto y Rango Etario
agrupado_edad = df.groupby(['Product Category', 'Rango_Etario'])['Total Amount'].agg(['sum', 'mean', 'count', 'min', 'max', 'std', 'var'])

print("Estadísticas por Producto y Rango Etario:")
display(agrupado_edad.round(2))

# Variables dinámicas para capturar los extremos del DataFrame agrupado_edad
segmento_fuerte = agrupado_edad['sum'].idxmax()
ticket_alto = agrupado_edad['mean'].idxmax()

print(
    f"=== Hallazgos Estratégicos: Segmentación por Rango Etario ===\n"
    f"1. Perfil de Ingresos: El segmento {segmento_fuerte[1]} en la categoría {segmento_fuerte[0]}\n"
    f"   representa el mayor volumen de ventas totales. Estrategia: Este es el 'core' del\n"
    f"   negocio; se deben priorizar canales de marketing que este grupo etario frecuente\n"
    f"   para asegurar la retención de este flujo de caja.\n\n"
    f"2. Poder Adquisitivo: El grupo {ticket_alto[1]} lidera el ticket promedio (mean) en\n"
    f"   {ticket_alto[0]}. Táctica: Aunque compren menos veces (count), su gasto unitario es\n"
    f"   superior. Resultados: Introducir productos 'Premium' o de gama alta dirigidos\n"
    f"   exclusivamente a este perfil para elevar el margen de utilidad por transacción.\n\n"
    f"3. Oportunidad de Crecimiento: Si el segmento 'Joven' muestra un conteo (count) alto pero\n"
    f"   un promedio bajo, se identifican consumidores frecuentes de bajo presupuesto.\n"
    f"   Táctica: Implementar modelos de suscripción o descuentos por volumen para capturar\n"
    f"   su lealtad desde una etapa temprana de su ciclo de vida como consumidor."
)

Estadísticas por Producto y Rango Etario:


sum    mean  count  min   max     std  \
Product Category Rango_Etario                                            
Beauty           Joven         31280  521.33     60   25  2000  619.75   
                 Adulto        59645  492.93    121   25  2000  593.32   
                 Senior        52590  417.38    126   25  2000  503.83   
Clothing         Joven         26510  519.80     51   25  2000  615.27   
                 Adulto        69525  463.50    150   25  2000  562.60   
                 Senior        59545  396.97    150   25  2000  514.05   
Electronics      Joven         26760  461.38     58   25  2000  573.94   
                 Adulto        61180  449.85    136   25  2000  581.51   
                 Senior        68965  465.98    148   25  2000  555.68   

                                     var  
Product Category Rango_Etario             
Beauty           Joven         384086.33  
                 Adulto        352023.61  
                 Senior        253849.49  
Clothing         Joven         378562.96  
                 Adulto        316516.36  
                 Senior        264243.59  
Electronics      Joven         329412.10  
                 Adulto        338158.50  
                 Senior        308775.05

=== Hallazgos Estratégicos: Segmentación por Rango Etario ===
1. Perfil de Ingresos: El segmento Adulto en la categoría Clothing
   representa el mayor volumen de ventas totales. Estrategia: Este es el 'core' del
   negocio; se deben priorizar canales de marketing que este grupo etario frecuente
   para asegurar la retención de este flujo de caja.

2. Poder Adquisitivo: El grupo Joven lidera el ticket promedio (mean) en
   Beauty. Táctica: Aunque compren menos veces (count), su gasto unitario es
   superior. Resultados: Introducir productos 'Premium' o de gama alta dirigidos
   exclusivamente a este perfil para elevar el margen de utilidad por transacción.

3. Oportunidad de Crecimiento: Si el segmento 'Joven' muestra un conteo (count) alto pero
   un promedio bajo, se identifican consumidores frecuentes de bajo presupuesto.
   Táctica: Implementar modelos de suscripción o descuentos por volumen para capturar
   su lealtad desde una etapa temprana de su ciclo de vida como consumidor.



3. Análisis Personalizado con apply
    * Función personalizada: Aplica funciones personalizadas para realizar análisis específicos que no se pueden lograr con las funciones de agregación estándar.
    * Ejemplo de uso avanzado: Calcula la desviación de cada venta respecto a la media de su grupo.
    


* Función de Desviación

In [12]:
# Aplicar función a cada categoría de producto
df['Desviacion_Media'] = df.groupby('Product Category')['Total Amount'].transform(lambda x: x - x.mean())

# Verificamos los resultados
print("Análisis de desviación respecto a la media por categoría:")
display(df[['Product Category', 'Total Amount', 'Desviacion_Media']].head(10).round(2))

# Variables para identificar extremos de desviación en el DataFrame
max_desv = df.loc[df['Desviacion_Media'].idxmax()]
min_desv = df.loc[df['Desviacion_Media'].idxmin()]

print(
    f"=== Hallazgos Estratégicos: Análisis de Desviación de Ventas ===\n"
    f"1. Identificación de Outliers: Se detectó una venta en {max_desv['Product Category']} con una\n"
    f"   desviación positiva de {max_desv['Desviacion_Media']:.2f}. Resultados: Estas transacciones\n"
    f"   'atípicas' representan clientes de alto valor o compras corporativas que no siguen\n"
    f"   el patrón común. Táctica: Crear un protocolo de atención VIP para estos casos.\n\n"
    f"2. Desempeño Bajo el Promedio: Las desviaciones negativas (como la de {min_desv['Desviacion_Media']:.2f})\n"
    f"   indican ventas que no alcanzaron el potencial de la categoría. Táctica: Analizar si\n"
    f"   estos productos necesitan un ajuste en el 'Product Mix' o si fueron afectados por\n"
    f"   descuentos agresivos que sacrificaron margen innecesariamente.\n\n"
    f"3. Estabilidad del Sistema: Si la mayoría de las desviaciones son cercanas a cero, la\n"
    f"   categoría es altamente predecible. Estrategia: Para el consumidor, esto significa\n"
    f"   precios estables; para la empresa, permite una planificación financiera con menor\n"
    f"   riesgo y un flujo de caja constante."
)

Análisis de desviación respecto a la media por categoría:


,Product Category,Total Amount,Desviacion_Media
0,Beauty,150,-317.48
1,Clothing,1000,556.75
2,Electronics,30,-428.79
3,Clothing,500,56.75
4,Beauty,100,-367.48
5,Beauty,30,-437.48
6,Clothing,50,-393.25
7,Electronics,100,-358.79
8,Electronics,600,141.21
9,Clothing,200,-243.25


=== Hallazgos Estratégicos: Análisis de Desviación de Ventas ===
1. Identificación de Outliers: Se detectó una venta en Clothing con una
   desviación positiva de 1556.75. Resultados: Estas transacciones
   'atípicas' representan clientes de alto valor o compras corporativas que no siguen
   el patrón común. Táctica: Crear un protocolo de atención VIP para estos casos.

2. Desempeño Bajo el Promedio: Las desviaciones negativas (como la de -442.48)
   indican ventas que no alcanzaron el potencial de la categoría. Táctica: Analizar si
   estos productos necesitan un ajuste en el 'Product Mix' o si fueron afectados por
   descuentos agresivos que sacrificaron margen innecesariamente.

3. Estabilidad del Sistema: Si la mayoría de las desviaciones son cercanas a cero, la
   categoría es altamente predecible. Estrategia: Para el consumidor, esto significa
   precios estables; para la empresa, permite una planificación financiera con menor
   riesgo y un flujo de caja constante.


* Función Bono de Lealtad

In [13]:
# Aplicar función a la columna Total Amount
# 'calcular_puntos' ya está disponible porque la extrajiste en la celda anterior
df['Puntos_Lealtad'] = df['Total Amount'].apply(calcular_puntos)

# Resultados
print("Cálculo de puntos por compra:")
print(df[['Total Amount', 'Puntos_Lealtad']].head(10))

# Variables para la interpretación dinámica
total_puntos = df['Puntos_Lealtad'].sum()
tasa_conversion = (df['Puntos_Lealtad'] > 0).mean() * 100

print(
    f"=== Hallazgos Estratégicos: Programa de Puntos de Lealtad ===\n"
    f"1. Alcance del Programa: El {tasa_conversion:.1f}% de las transacciones generaron puntos.\n"
    f"   Estrategia: Este indicador mide la efectividad de los umbrales de recompensa.\n"
    f"   Si la tasa es muy baja, los niveles actuales pueden resultar inalcanzables\n"
    f"   para el consumidor promedio, desincentivando la participación.\n\n"
    f"2. Incentivo al Ticket Promedio: La función aplica un beneficio escalonado (hasta 15%).\n"
    f"   Táctica: El objetivo es el 'Upselling'. Si un cliente está cerca de los $750 o\n"
    f"   $1000, el sistema debe sugerir productos adicionales (bundles) para alcanzar\n"
    f"   el siguiente nivel de puntos, aumentando el ingreso por sesión.\n\n"
    f"3. Pasivo Financiero: Se han generado un total de {total_puntos:.2f} puntos. Resultados:\n"
    f"   Para la empresa, estos puntos representan un compromiso futuro. Táctica:\n"
    f"   Monitorear la tasa de redención (burn rate) para asegurar que el costo del\n"
    f"   programa no erosione el margen de utilidad neta en el largo plazo."
)
print()
# Análisis de conveniencia financiera
ventas_con_puntos = df[df['Puntos_Lealtad'] > 0]
costo_total_puntos = df['Puntos_Lealtad'].sum()
ingreso_total = df['Total Amount'].sum()
porcentaje_costo_sobre_venta = (costo_total_puntos / ingreso_total) * 100

print(
    f"=== Análisis de Viabilidad Económica ===\n"
    f"1. Costo del Programa: El costo total en puntos es de ${costo_total_puntos:,.2f}.\n"
    f"   Esto representa un {porcentaje_costo_sobre_venta:.2f}% de la venta bruta total. Si el margen\n"
    f"   de utilidad neta de la empresa es superior al 20%, el programa es financieramente\n"
    f"   sostenible bajo el modelo de 'quema de puntos' controlada.\n\n"
    f"2. Eficiencia del Ticket: Las ventas que generan el 15% de puntos (>1000) son las más\n"
    f"   costosas. Resultados: Si este grupo no muestra una recurrencia (count) significativamente\n"
    f"   mayor que los demás, se está 'regalando' margen a compras que serían de alto valor\n"
    f"   de todas formas. Táctica: Ajustar el tope máximo de puntos por transacción.\n\n"
    f"3. Punto de Equilibrio (Breakeven): Para que el programa sea conveniente, el aumento en\n"
    f"   la frecuencia de compra debe compensar el {porcentaje_costo_sobre_venta:.2f}% cedido en puntos.\n"
    f"   Estrategia: El programa es conveniente si logra reducir el CAC (Costo de Adquisición\n"
    f"   de Clientes) mediante la retención orgánica de los usuarios actuales."
)

Cálculo de puntos por compra:
   Total Amount  Puntos_Lealtad
0           150             0.0
1          1000           100.0
2            30             0.0
3           500            25.0
4           100             0.0
5            30             0.0
6            50             0.0
7           100             0.0
8           600            30.0
9           200            10.0
=== Hallazgos Estratégicos: Programa de Puntos de Lealtad ===
1. Alcance del Programa: El 45.8% de las transacciones generaron puntos.
   Estrategia: Este indicador mide la efectividad de los umbrales de recompensa.
   Si la tasa es muy baja, los niveles actuales pueden resultar inalcanzables
   para el consumidor promedio, desincentivando la participación.

2. Incentivo al Ticket Promedio: La función aplica un beneficio escalonado (hasta 15%).
   Táctica: El objetivo es el 'Upselling'. Si un cliente está cerca de los $750 o
   $1000, el sistema debe sugerir productos adicionales (bundles) para alcanzar
   el s

4. Documentación
    * Comentarios claros: Documenta claramente cada paso del análisis, explicando qué se hizo y por qué se hizo.
    * Código legible: Asegúrate de que el código sea legible y esté bien comentado.

In [14]:
print("=== Documentación ===")
print()
print(
    f" Se usó f-strings para los comentarios en funciones (archivo .py). \n"
    f" Se apoyó en IA para generar interpretaciones más profundas dado el poco conociiento del campo del problema.\n"
)

=== Documentación ===

 Se usó f-strings para los comentarios en funciones (archivo .py). 
 Se apoyó en IA para generar interpretaciones más profundas dado el poco conociiento del campo del problema.



** Guardar Dataframe

In [ ]:
# Guardar el DataFrame resultante como CSV
# Usamos index=False para que no cree una columna extra de números al principio
df.to_csv('retail_sales_final_P3.csv', index=False)

print("Archivo guardado exitosamente como 'retail_sales_final_P3.csv'")